In [1]:
from google.colab import drive, userdata

userdata.get('HF_TOKEN')
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import zipfile

!unzip /content/drive/MyDrive/data_preprocessed_llava.zip

Archive:  /content/drive/MyDrive/data_preprocessed_llava.zip
   creating: data_preprocessed_llava/
  inflating: data_preprocessed_llava/dataset_dict.json  
   creating: data_preprocessed_llava/train/
  inflating: data_preprocessed_llava/train/data-00000-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00001-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00002-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00003-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00004-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00005-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00006-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00007-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00008-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00009-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00010-of-00021.arrow  
  inflating: data_p

# Training

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from datasets import load_from_disk
from transformers import AutoProcessor, LlavaModel
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datetime import datetime
import os

In [4]:
# -----------------------------
# Config
# -----------------------------
DATAPATH = "./data_preprocessed_llava"
BATCH_SIZE = 8
NUM_EPOCHS = 15
LR = 5e-5
WEIGHT_DECAY = 0.01
device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# Load dataset
# -----------------------------
print("loading from disk")
data = load_from_disk(DATAPATH)
data.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "pixel_values", "labels"]
)

train_loader = DataLoader(data["train"], batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(data["validation"], batch_size=BATCH_SIZE, num_workers=0)


loading from disk


Loading dataset from disk:   0%|          | 0/21 [00:00<?, ?it/s]

In [5]:
!pip install -U bitsandbytes>=0.46.1

In [6]:
# -----------------------------
# Load LLaVA model + processor
# docs: https://huggingface.co/docs/transformers/en/model_doc/llava
# -----------------------------

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
llava = LlavaModel.from_pretrained(
    "llava-hf/llava-1.5-7b-hf",
    quantization_config=bnb_config,
    device_map="auto"
)

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/685 [00:00<?, ?it/s]

LlavaModel LOAD REPORT from: llava-hf/llava-1.5-7b-hf
Key                           | Status     |  | 
------------------------------+------------+--+-
language_model.lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
llava

LlavaModel(
  (vision_tower): CLIPVisionModel(
    (vision_model): CLIPVisionTransformer(
      (embeddings): CLIPVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
        (position_embedding): Embedding(577, 1024)
      )
      (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-23): 24 x CLIPEncoderLayer(
            (self_attn): CLIPAttention(
              (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
              (v_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
              (q_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
              (out_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            )
            (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn): QuickGEL

In [8]:
import torch
import torch.nn as nn

class LLaVAClassifier(nn.Module):
    def __init__(self, llava_model, hidden_size=4096):
        super().__init__()
        self.llava = llava_model

        # Add a linear classifier head
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size, eps=1e-5, elementwise_affine=True),
            nn.Linear(hidden_size, 1024),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(1024, 1)   # binary classification
        ).float()

    def forward(self, pixel_values, input_ids, attention_mask):
        # Forward pass through LLaVA
        outputs = self.llava(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            return_dict=True,
            output_hidden_states=True,
        )

        # Use the last token of the last hidden state
        # shape: (batch_size, seq_len, hidden_dim)
        last_hidden = outputs.hidden_states[-1].mean(dim=1).float()
        logits = self.classifier(last_hidden).squeeze(-1)
        return logits


In [9]:
# -----------------------------
# Freeze all backbone parameters
# -----------------------------
for param in llava.parameters():
    param.requires_grad = False

In [10]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,                          # rank — start here, tune down to 8 or 4
    lora_alpha=32,                 # scaling factor (alpha/r = 2 is a safe default)
    target_modules=[
        "multi_modal_projector.linear_1",
        "multi_modal_projector.linear_2",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,   # not seq2seq or causal LM
)

llava = get_peft_model(llava, lora_config)
llava.print_trainable_parameters()

trainable params: 212,992 || all params: 6,932,305,920 || trainable%: 0.0031


In [11]:
# Wrap with classifier
model = LLaVAClassifier(llava, hidden_size=llava.config.text_config.hidden_size).to(device)

# -----------------------------
# Compute class imbalance for pos_weight
# -----------------------------
labels = data["train"]["labels"]
pos_count = sum(labels)
neg_count = len(labels) - pos_count
pos_weight = neg_count / max(pos_count, 1)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=device, dtype=torch.float16))
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

/tmp/ipykernel_1231/737160240.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=device, dtype=torch.float16))


In [12]:
import os
import time

os.makedirs("checkpoints", exist_ok=True)

# -----------------------------
# Training loop
# -----------------------------
best_acc = 0
train_losses, val_losses, val_accs = [], [], []

scaler = GradScaler()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Train Epoch {epoch}", leave=False):
        pixel_values = batch["pixel_values"].to(device, dtype=torch.float16)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device, dtype=torch.float32)

        optimizer.zero_grad()
        with autocast(device, dtype=torch.float16):
            logits = model(pixel_values, input_ids, attention_mask)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch} Train Loss: {train_loss}")
    train_losses.append(train_loss)

    # -----------------------------
    # Validation
    # -----------------------------
    model.eval()
    val_loss = 0
    correct, total = 0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Val Epoch {epoch}", leave=False):
            pixel_values = batch["pixel_values"].to(device, dtype=torch.float16)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device, dtype=torch.float32)

            with autocast(device, dtype=torch.float16):
                logits = model(pixel_values, input_ids, attention_mask)
                loss = criterion(logits, labels)

            val_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    print(f"Epoch {epoch} Validation Loss: {val_loss}")
    print(f"Epoch {epoch} Validation Accuracy: {val_acc}")
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    epoch_mins = (time.time() - epoch_start) / 60
    print(f"Epoch {epoch} Time: {epoch_mins:.2f} min")

    if val_acc > best_acc:
      best_acc = val_acc
      torch.save(model.state_dict(), f"checkpoints/best_model_epoch_{epoch}.pt")
      print("Saved best model!")

torch.save(model.state_dict(), f"checkpoints/epoch_{epoch}.pt")

Epoch 0 Train Loss: 0.8745941803435422


Epoch 0 Validation Loss: 0.8911974296391567
Epoch 0 Validation Accuracy: 0.6647058823529411
Epoch 0 Time: 28.82 min
Saved best model!


Epoch 1 Train Loss: 0.8204720374371938


Epoch 1 Validation Loss: 0.8461064509699278
Epoch 1 Validation Accuracy: 0.6564705882352941
Epoch 1 Time: 28.79 min


Epoch 2 Train Loss: 0.7704972916885104


...

Epoch 12 Validation Loss: 1.982534035204727
Epoch 12 Validation Accuracy: 0.6223529411764706
Epoch 12 Time: 28.64 min


Epoch 13 Train Loss: 0.28976876274627383


Epoch 13 Validation Loss: 2.8531729832430868
Epoch 13 Validation Accuracy: 0.6282352941176471
Epoch 13 Time: 28.61 min


Epoch 14 Train Loss: 0.3677358469785803


Epoch 14 Validation Loss: 3.4632394948852396
Epoch 14 Validation Accuracy: 0.6058823529411764
Epoch 14 Time: 28.55 min


In [15]:
# -----------------------------
# Save outputs
# -----------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_root = "outputs"
run_dir = os.path.join(output_root, f"llava_{timestamp}")
os.makedirs(run_dir, exist_ok=True)
epochs = range(1, len(val_losses) + 1)

# Save metrics
metrics = {
    "train_losses": train_losses,
    "val_losses": val_losses,
    "val_accs": val_accs,
    "epochs": list(epochs),
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY
}
with open(os.path.join(run_dir, "finetune_metrics.json"), "w") as mf:
    import json
    json.dump(metrics, mf, indent=2)

# Loss figure (train & validation)
fig_loss = plt.figure(figsize=(6, 4))
plt.plot(epochs, train_losses, marker='o', label='Train Loss')
plt.plot(epochs, val_losses, marker='o', label='Val Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
loss_filename = os.path.join(run_dir, f"loss.png")
plt.savefig(loss_filename)
plt.close(fig_loss)

# Validation accuracy figure
fig_acc = plt.figure(figsize=(6, 4))
plt.plot(epochs, val_accs, marker='o')
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.tight_layout()
acc_filename = os.path.join(run_dir, f"val_acc.png")
plt.savefig(acc_filename)
plt.close(fig_acc)


In [17]:
# Save model
torch.save(model.state_dict(), os.path.join(run_dir, "classifier.pt"))
processor.save_pretrained(os.path.join(run_dir))
torch.save(model, "classifier_full.pt")


print("Training complete, model saved at", run_dir)

Training complete, model saved at outputs/llava_20260504_031821


In [ ]:
model.classifier.save()

In [20]:
torch.save(model.state_dict(), os.path.join("/content/drive/MyDrive/", "classifier.pt"))


In [24]:
lora_trained_model_11pm = model

In [27]:
!ls -aslh /content/outputs/llava_20260504_031821/classifier.pt

3.6G -rw-r--r-- 1 root root 3.6G May  4 03:21 /content/outputs/llava_20260504_031821/classifier.pt


In [28]:
!cp /content/outputs/llava_20260504_031821/classifier.pt /content/drive/MyDrive/classifier.pt

# Evaluation

In [39]:
import os
import glob
import json
from datetime import datetime
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_from_disk
from transformers import LlavaModel, AutoProcessor
from tqdm import tqdm
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# run_dir = "/content/outputs/llava_20260504_031821"
# datapath="./data_preprocessed_llava"
# output_root="outputs"
# batch_size=64

# device = "cuda" if torch.cuda.is_available() else "cpu"

# data = load_from_disk(datapath)
# data.set_format("torch") # ensure tensors

# val_dataset = data["validation"]
# val_loader = DataLoader(val_dataset, batch_size=batch_size)

In [42]:
gc.collect()
torch.cuda.empty_cache()

In [43]:
def evaluate(
    run_dir,
    datapath="./data_preprocessed_llava",
    batch_size=64,
    output_root="outputs",
    checkpoint="classifier.pt",
):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    data = load_from_disk(datapath)
    data.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "pixel_values", "labels"]
    )
    val_loader = DataLoader(data["validation"], batch_size=batch_size, num_workers=0)

    full_run_dir = os.path.join(output_root, run_dir)
    classifier_path = os.path.join(full_run_dir, checkpoint)
    if not os.path.exists(classifier_path):
        raise FileNotFoundError(f"Checkpoint not found: {classifier_path}")

    print(f"Loading processor and base LLaVA to device={device}...")
    processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    llava = LlavaModel.from_pretrained(
        "llava-hf/llava-1.5-7b-hf",
        quantization_config=bnb_config,
        device_map="auto",
    )

    # Freeze backbone (mirrors training setup before LoRA)
    for param in llava.parameters():
        param.requires_grad = False

    # Re-apply same LoRA config as training
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=[
            "multi_modal_projector.linear_1",
            "multi_modal_projector.linear_2",
        ],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.FEATURE_EXTRACTION,
    )
    llava = get_peft_model(llava, lora_config)

    # Wrap with classifier head
    hidden_size = llava.config.text_config.hidden_size
    model = LLaVAClassifier(llava, hidden_size=hidden_size).to(device)

    # Load saved weights
    print(f"Loading checkpoint from {classifier_path} ...")
    state = torch.load(classifier_path, map_location=device, weights_only=False)
    try:
        model.load_state_dict(state)
    except Exception:
        model.load_state_dict(state, strict=False)
        print("Warning: loaded with strict=False — some keys may not have matched.")

    model.eval()

    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Eval", leave=False):
            pixel_values = batch["pixel_values"].to(device, dtype=torch.float16)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device, dtype=torch.float32)

            logits = model(pixel_values, input_ids, attention_mask)
            if logits.dim() == 2 and logits.size(1) == 1:
                logits = logits.squeeze(1)

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).long()

            y_true.extend(labels.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())
            y_prob.extend(probs.cpu().float().numpy().tolist())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    acc = accuracy_score(y_true, y_pred)
    prec, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    conf = confusion_matrix(y_true, y_pred).tolist()

    roc_auc = None
    if len(np.unique(y_true)) == 2:
        try:
            roc_auc = float(roc_auc_score(y_true, y_prob))
        except Exception:
            pass

    metrics = {
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(recall),
        "f1": float(f1),
        "confusion_matrix": conf,
        "roc_auc": roc_auc,
        "n_samples": int(len(y_true)),
    }

    os.makedirs(full_run_dir, exist_ok=True)
    out_json = os.path.join(full_run_dir, "eval_metrics.json")
    with open(out_json, "w") as f:
        json.dump(metrics, f, indent=2)

    # ROC curve
    try:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        fig = plt.figure(figsize=(6, 6))
        plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.4f})" if roc_auc else "ROC curve")
        plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve")
        plt.legend(loc="lower right")
        plt.tight_layout()
        fig.savefig(os.path.join(full_run_dir, "roc_curve.png"))
        plt.close(fig)
        print(f"Saved ROC curve to {full_run_dir}/roc_curve.png")
    except Exception as e:
        print(f"Warning: failed to save ROC curve: {e}")

    # Confusion matrix
    try:
        cm = np.array(conf)
        fig = plt.figure(figsize=(6, 6))
        ax = fig.add_subplot(1, 1, 1)
        im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
        ax.figure.colorbar(im, ax=ax)
        classes = ["0", "1"]
        tick_marks = np.arange(len(classes))
        ax.set_xticks(tick_marks)
        ax.set_yticks(tick_marks)
        ax.set_xticklabels(classes)
        ax.set_yticklabels(classes)
        plt.xlabel("Predicted label")
        plt.ylabel("True label")
        plt.title("Confusion Matrix")
        thresh = cm.max() / 2.0 if cm.size else 0
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, format(int(cm[i, j]), 'd'),
                        ha="center", va="center",
                        color="white" if cm[i, j] > thresh else "black")
        plt.tight_layout()
        fig.savefig(os.path.join(full_run_dir, "confusion_matrix.png"))
        plt.close(fig)
        print(f"Saved confusion matrix to {full_run_dir}/confusion_matrix.png")
    except Exception as e:
        print(f"Warning: failed to save confusion matrix: {e}")

    # Score distribution
    try:
        fig = plt.figure(figsize=(6, 4))
        bins = np.linspace(0.0, 1.0, 21)
        plt.hist(y_prob[y_true == 0], bins=bins, density=True, alpha=0.6, label="neg (0)", color="C0", edgecolor="black")
        plt.hist(y_prob[y_true == 1], bins=bins, density=True, alpha=0.6, label="pos (1)", color="C1", edgecolor="black")
        plt.xlabel("Predicted probability (positive class)")
        plt.ylabel("Density")
        plt.title("Score Distribution by True Label")
        plt.legend(loc="upper center")
        plt.tight_layout()
        fig.savefig(os.path.join(full_run_dir, "score_distribution.png"))
        plt.close(fig)
        print(f"Saved score distribution to {full_run_dir}/score_distribution.png")
    except Exception as e:
        print(f"Warning: failed to save score distribution: {e}")

    print("\nEvaluation summary:")
    print(f"  samples:          {metrics['n_samples']}")
    print(f"  accuracy:         {metrics['accuracy']:.4f}")
    print(f"  precision:        {metrics['precision']:.4f}")
    print(f"  recall:           {metrics['recall']:.4f}")
    print(f"  f1:               {metrics['f1']:.4f}")
    if roc_auc is not None:
        print(f"  roc_auc:          {roc_auc:.4f}")
    print(f"  confusion_matrix: {metrics['confusion_matrix']}")
    print(f"Saved metrics to {out_json}")


In [44]:
evaluate(
    datapath="./data_preprocessed_llava",
    batch_size=8,
    run_dir="llava_20260504_031821",
    output_root="outputs",
    checkpoint="classifier.pt",
)

Loading dataset from disk:   0%|          | 0/21 [00:00<?, ?it/s]

Loading processor and base LLaVA to device=cuda...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/685 [00:00<?, ?it/s]

LlavaModel LOAD REPORT from: llava-hf/llava-1.5-7b-hf
Key                           | Status     |  | 
------------------------------+------------+--+-
language_model.lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading checkpoint from outputs/llava_20260504_031821/classifier.pt ...


Saved ROC curve to outputs/llava_20260504_031821/roc_curve.png
Saved confusion matrix to outputs/llava_20260504_031821/confusion_matrix.png
Saved score distribution to outputs/llava_20260504_031821/score_distribution.png

Evaluation summary:
  samples:          850
  accuracy:         0.5988
  precision:        0.4832
  recall:           0.6626
  f1:               0.5589
  roc_auc:          0.6709
  confusion_matrix: [[293, 231], [110, 216]]
Saved metrics to outputs/llava_20260504_031821/eval_metrics.json
